In [ ]:
# Repository-relative paths for the anonymized reproduction package.
import os
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'analysis').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from within the repository tree.')

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
MODULE_DIR = REPO_ROOT / 'analysis' / '05_be_meta_regression'
EXTERNAL_DATA_ROOT = Path(os.environ.get('HEATPA_DATA_ROOT', REPO_ROOT / 'external_data'))


In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle


# =============================================================================
# 1. 路径设置
# =============================================================================
# 截图一：本代码使用已经整理好的 Source Data 文件夹作为唯一数据源。
SOURCE_DATA_DIR = MODULE_DIR / "data" / "figure5_ab"

# 截图二：PNG / SVG 图件输出文件夹。
FIGURE_OUTPUT_DIR = MODULE_DIR / "output"

# 截图三：右侧表格结果单独导出的 CSV 文件夹。
TABLE_CSV_DIR = MODULE_DIR / "output"

# 核心森林图源数据表。
SOURCE_TABLE = SOURCE_DATA_DIR / "be_meta_predictors_grouped_source_records_final_no_crime_no_bh.csv"

# BD/BH 审计文件；若不存在则自动生成空审计文件，不影响绘图。
SOURCE_AUDIT_PATHS = SOURCE_DATA_DIR / "bd_bh_source_audit_paths.csv"

# 已归档的 panel a/b 源记录文件；仅用于备份复制，不参与重新计算。
SOURCE_PANEL_C = SOURCE_DATA_DIR / "panel_c_mean_no_crime_activity_3plus1_national_cehwi_composite_records.csv"
SOURCE_PANEL_D = SOURCE_DATA_DIR / "panel_d_gini_no_crime_activity_3plus1_national_cehwi_composite_records.csv"

# lag-response 文件当前森林图不直接使用，仅用于结果归档。
LAG_PROFILE_FILES = [
    SOURCE_DATA_DIR / "pooled_lag_response_p25.csv",
    SOURCE_DATA_DIR / "pooled_lag_response_p50.csv",
    SOURCE_DATA_DIR / "pooled_lag_response_p75.csv",
    SOURCE_DATA_DIR / "pooled_lag_response_p90.csv",
    SOURCE_DATA_DIR / "pooled_lag_response_p95.csv",
]

# 输出组织：图件直接保存至截图二；中间表、代码和报告保存至其子文件夹。
OUT_ROOT = FIGURE_OUTPUT_DIR
FIG_DIR = FIGURE_OUTPUT_DIR
DATA_DIR = FIGURE_OUTPUT_DIR / "data"
CODE_DIR = FIGURE_OUTPUT_DIR / "code"
REPORT_DIR = FIGURE_OUTPUT_DIR / "report"


# =============================================================================
# 2. 可视化参数：所有图形布局、字体、线宽、颜色均集中在这里修改
# =============================================================================
# -----------------------------
# 2.1 字体
# -----------------------------
LEFT_FONT_FAMILY = "sans-serif"
LEFT_SANS_FONTS = ["Arial", "DejaVu Sans", "Liberation Sans"]
TABLE_FONT_FAMILY = "Times New Roman"
TABLE_FALLBACK_SERIF_FONTS = ["Times New Roman", "Times", "DejaVu Serif", "Liberation Serif"]
SVG_FONT_AS_TEXT = True

GLOBAL_FONT_SIZE = 8.5
TITLE_FONT_SIZE = 14.0
SUBTITLE_FONT_SIZE = 9.0
HEADER_FONT_SIZE = 9.0
LABEL_FONT_SIZE = 9.4
TABLE_FONT_SIZE = 8.5
AXIS_FONT_SIZE = 9.2
TICK_FONT_SIZE = 9.0
LEGEND_FONT_SIZE = 8.5
STAR_FONT_SIZE = 7.6

TITLE_FONT_WEIGHT = "bold"
HEADER_FONT_WEIGHT = "bold"
HIGHLIGHT_LABEL_FONT_WEIGHT = "bold"
NORMAL_LABEL_FONT_WEIGHT = "normal"

# -----------------------------
# 2.2 图件尺寸与边距
# -----------------------------
# 宽度固定，高度可根据右侧表格所需高度自适应。
FIG_WIDTH = 12.6
AUTO_FIG_HEIGHT = True
FIXED_FIG_HEIGHT = 7.0
HEIGHT_PER_PREDICTOR = 0.66
HEIGHT_EXTRA_FOR_TITLE_LEGEND_AXIS = 1.35
MIN_FIG_HEIGHT = 5.8
MAX_FIG_HEIGHT = 9.2

EXPORT_DPI = 600
SAVE_BBOX_TIGHT = True
EXPORT_PAD_INCHES = 0.02

# figure 内部绘图区边距。通过这些参数可控制标题、图例和底部轴名的空间。
GRID_LEFT = 0.035
GRID_RIGHT = 0.992
GRID_TOP = 0.865
GRID_BOTTOM = 0.095
GRID_WSPACE = 0.040

TITLE_X = 0.035
TITLE_Y = 0.960
SUBTITLE_X = 0.035
SUBTITLE_Y = 0.918

LEGEND_X = 0.985
LEGEND_Y = 0.938
LEGEND_NCOL = 4
LEGEND_HANDLE_LENGTH = 1.8
LEGEND_COLUMN_SPACING = 0.95
LEGEND_MARKER_SIZE = 4.8
LEGEND_LINE_WIDTH = 1.45

# -----------------------------
# 2.3 宽度比例控制参数
# -----------------------------
# 参数 1：左侧图内部“行名列 : 森林图列”的宽度比例。
# 目标：压缩行名空间、增加森林图空间。
# 例如 (0.24, 0.76) 表示左侧图中 24% 给 Predictor 行名，76% 给森林图。
PREDICTOR_TO_FOREST_WIDTH_RATIO = (0.28, 0.72)

# 参数 2：左侧图整体 : 右侧表格 的宽度比例。
# 目标：增加左侧图宽度、压缩右侧表格宽度。
# 左侧图整体 = Predictor 行名列 + 森林图列。
# 例如 (1.30, 0.70) 表示左侧图整体明显宽于右侧表格。
LEFT_PLOT_TO_RIGHT_TABLE_WIDTH_RATIO = (1.2, 0.80)

# 如需更接近原图，可改为：
# PREDICTOR_TO_FOREST_WIDTH_RATIO = (0.34, 0.66)
# LEFT_PLOT_TO_RIGHT_TABLE_WIDTH_RATIO = (1.00, 1.00)


# -----------------------------
# 2.4 表格行距、自适应 y 轴范围和灰白背景
# -----------------------------
# 每个 Predictor 大类之间的中心间距。右侧每个大类内有四行活动类型。
ROW_STEP = 1.00
ACTIVITY_OFFSETS = {
    "all": -0.33,
    "ride": -0.11,
    "run": 0.11,
    "walk": 0.33,
}

# y 轴范围控制。
# BOTTOM_ALIGN_TO_DATA_EDGE=True 时，绘图区底部直接贴着最底部 Predictor 分组边界，
# 可显著减少右侧表格和左侧森林图下方的空白。
BOTTOM_ALIGN_TO_DATA_EDGE = True
BOTTOM_DATA_EDGE_PAD = 0.000

# 数据区顶部边界在轴域中的相对位置；需小于 HEADER_BAND_BOTTOM，避免顶部分组压到表头。
TOP_DATA_EDGE_AXES_FRAC = 0.918

# 若 BOTTOM_ALIGN_TO_DATA_EDGE=False，则回退为按最低/最高活动文字反算 y 轴范围。
BOTTOM_ACTIVITY_AXES_FRAC = 0.030
TOP_ACTIVITY_AXES_FRAC = 0.895

# 交替灰白背景覆盖每个 Predictor 的完整分组区域；横线画在分组边界上，因此不会再出现边界与横线错位。
SHOW_ALT_ROW_BANDS = True
ALT_ROW_START_WITH_GREY = True
SHOW_TABLE_GROUP_SEPARATOR = True

# 灰白背景绘制方式：
# "figure" = 用一个跨越左侧标签、森林图和右侧表格的 figure 级矩形绘制，
#            避免多个 Axes 各自 axhspan 在导出/缩放时产生横向灰线或接缝。
# "axes"   = 回退到每个 Axes 内分别绘制灰白背景。
ROW_BAND_DRAW_MODE = "figure"
ROW_BAND_ZORDER = -30
AXES_FACE_COLOR = "none"

# figure 级背景矩形左右边缘微调；一般保持 0 即可。
ROW_BAND_X_PAD_FIG = 0.000

# 灰白背景上下边界微重叠，避免 PDF/PNG 缩放时出现 1 像素细缝。
# 数值为数据坐标单位，建议 0.000~0.006。
ROW_BAND_Y_OVERLAP = 0.004

# 表头灰底在轴域坐标中的位置。
HEADER_BAND_BOTTOM = 0.935
HEADER_BAND_TOP = 1.000
HEADER_Y = 0.966

# -----------------------------
# 2.5 右侧表格列宽与列位置
# -----------------------------
# 右侧表格三列宽度比例。
# 当前设置为 1:1:1，即 Activity、beta [95% CI]、p 三列均分整个右侧表格宽度。
# 如需恢复不均分，可改为例如 (0.25, 0.55, 0.20)。
TABLE_COLUMN_WIDTH_RATIOS = (1.0, 1.0, 1.0)

# 每列内部文字距离该列左边界的相对内边距，单位为右侧表格轴域宽度。
# Activity 和 beta 两列左对齐时使用；p 列默认居中。
TABLE_COLUMN_INNER_PAD = 0.025

# 表头是否居中放置在各自列内。True 更能体现“三列均分”的视觉效果。
TABLE_HEADER_CENTERED_IN_EQUAL_COLUMNS = True

# 数据文字对齐方式。
# 若希望所有数据也居中，可将 Activity 和 beta 改为 "center"。
TABLE_ACTIVITY_HA = "left"
TABLE_BETA_HA = "left"
TABLE_P_HA = "center"

# -----------------------------
# 2.6 线宽、点大小和森林图细节
# -----------------------------
AXES_LINE_WIDTH = 0.90
ZERO_LINE_WIDTH = 0.85
ZERO_LINE_STYLE = (0, (4, 3))
GRID_LINE_WIDTH = 0.60
CI_LINE_WIDTH = 1.55
CI_LINE_ALPHA = 0.88
POINT_SIZE = 32
POINT_EDGE_WIDTH = 0.55
STAR_X_OFFSET_FRAC = 0.018
STAR_Y_OFFSET = 0.025

GROUP_SEPARATOR_LW = 0.82
MAJOR_RULE_LW = 1.45
MAJOR_RULE_HEADER_Y = HEADER_BAND_BOTTOM
MAJOR_RULE_BOTTOM_Y = 0.000

# -----------------------------
# 2.7 颜色
# -----------------------------
TABLE_TEXT_COLOR = "#000000"
HEADER_TEXT_COLOR = "#000000"
PREDICTOR_TEXT_COLOR = "#1F1F1F"
SUBTITLE_COLOR = "#555555"
ZERO_LINE_COLOR = "#777777"
GRID_COLOR = "#E9E4DE"
ALT_ROW_COLOR = "#F7F7F7"
HEADER_BAND_COLOR = "#F2F2F2"
GROUP_SEPARATOR_COLOR = "#A6A6A6"
MAJOR_RULE_COLOR = "#7F7F7F"
POINT_EDGE_COLOR = "white"

# -----------------------------
# 2.8 文本内容开关
# -----------------------------
SHOW_SUBTITLE = True
SHOW_LEGEND = True
SHOW_SIGNIFICANCE_STARS_ON_FOREST = True
HIGHLIGHT_VARIABLES = {"Building Density"}


# =============================================================================
# 3. 数据和标签设置
# =============================================================================
MODES = {
    "MEAN_NO_CRIME_ACTIVITY_3PLUS1": "Mean predictors",
    "GINI_NO_CRIME_ACTIVITY_3PLUS1": "Gini predictors",
}

ACTIVITY_ORDER = ["all", "ride", "run", "walk"]
ACTIVITY_LABELS = {
    "all": "All activity",
    "ride": "Cycling",
    "run": "Running",
    "walk": "Walking",
}
ACTIVITY_COLORS = {
    "all": "#4D4D4D",
    "ride": "#A02D2B",
    "run": "#D99066",
    "walk": "#4C8EBA",
}

VARIABLE_ORDER = [
    "Urbanization Rate",
    "Population (20-55)",
    "Building Density",
    "Unemployment",
    "NDVI",
    "GDP",
    "Street Intersection Density",
    "Walkability Index",
]


# =============================================================================
# 4. 基础函数
# =============================================================================
def apply_style() -> None:
    """设置全局 matplotlib 风格。右侧表格会在 text 中单独指定 Times New Roman。"""
    plt.rcParams["font.family"] = LEFT_FONT_FAMILY
    plt.rcParams["font.sans-serif"] = LEFT_SANS_FONTS
    plt.rcParams["font.serif"] = TABLE_FALLBACK_SERIF_FONTS
    plt.rcParams["svg.fonttype"] = "none" if SVG_FONT_AS_TEXT else "path"
    plt.rcParams["font.size"] = GLOBAL_FONT_SIZE
    plt.rcParams["axes.spines.right"] = False
    plt.rcParams["axes.spines.top"] = False
    plt.rcParams["axes.linewidth"] = AXES_LINE_WIDTH
    plt.rcParams["legend.frameon"] = False


def figure_size(n_predictors: int) -> tuple[float, float]:
    """根据右侧表格的大类数量自适应图件高度，避免文字重叠和底部留白过多。"""
    if not AUTO_FIG_HEIGHT:
        return FIG_WIDTH, FIXED_FIG_HEIGHT
    height = HEIGHT_EXTRA_FOR_TITLE_LEGEND_AXIS + n_predictors * HEIGHT_PER_PREDICTOR
    height = max(MIN_FIG_HEIGHT, min(MAX_FIG_HEIGHT, height))
    return FIG_WIDTH, height


def drop_direction_columns(df: pd.DataFrame) -> pd.DataFrame:
    """删除 direction 相关数据列，避免输出数据表和右侧表格继续出现 direction。"""
    drop_cols = [c for c in df.columns if str(c).lower() in {"direction", "direction_significance"}]
    if drop_cols:
        return df.drop(columns=drop_cols)
    return df


def validate_inputs() -> None:
    """运行前检查必要输入文件，路径错误时给出清晰提示。"""
    required = [SOURCE_TABLE]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Required repository inputs are missing; verify the module-relative data paths:\n" + "\n".join(missing)
        )

    # lag-response 文件并非当前森林图绘制的必要输入，因此只提醒，不中断运行
    missing_lag = [p for p in LAG_PROFILE_FILES if not p.exists()]
    if missing_lag:
        print("Warning: 以下 fig5d lag-response 文件未找到，森林图仍可继续绘制：")
        for p in missing_lag:
            print(f"  - {p}")


def fmt_num(value: float, digits: int = 2) -> str:
    if not np.isfinite(value):
        return "NA"
    if abs(value) >= 100:
        return f"{value:.1f}"
    if abs(value) < 0.01 and value != 0:
        return f"{value:.2e}"
    return f"{value:.{digits}f}"


def fmt_p(value: float) -> str:
    if not np.isfinite(value):
        return "NA"
    if value < 0.001:
        return "<0.001"
    return f"{value:.3f}"


def sig_label(p_value: float, significant: object, star_label: object) -> str:
    star = "" if pd.isna(star_label) else str(star_label).strip()
    if star and star.lower() != "nan":
        return star
    if np.isfinite(p_value):
        if p_value < 0.001:
            return "***"
        if p_value < 0.01:
            return "**"
        if p_value < 0.05:
            return "*"
    sig = str(significant).lower()
    return "*" if sig == "sig" else "ns"


def prepare_panel_data(mode: str) -> pd.DataFrame:
    df = pd.read_csv(SOURCE_TABLE)
    sub = df[
        df["mode"].astype(str).eq(mode)
        & df["partition"].astype(str).eq("NATIONAL")
        & df["indicator"].astype(str).eq("cehwi")
        & df["base_model"].astype(str).eq("composite")
    ].copy()
    for col in ["coefficient", "ci_low", "ci_high", "p_value", "se", "z_value"]:
        if col in sub.columns:
            sub[col] = pd.to_numeric(sub[col], errors="coerce")
    sub = sub[sub["activity_type"].astype(str).isin(ACTIVITY_ORDER)]
    sub = sub[sub["variable_only"].astype(str).isin(VARIABLE_ORDER)]
    sub["variable_only"] = pd.Categorical(sub["variable_only"], VARIABLE_ORDER[::-1], ordered=True)
    sub["activity_type"] = pd.Categorical(sub["activity_type"], ACTIVITY_ORDER, ordered=True)
    sub = sub.sort_values(["variable_only", "activity_type"]).reset_index(drop=True)
    sub["significance_label"] = [
        sig_label(p, s, st) for p, s, st in zip(sub["p_value"], sub["significant"], sub["star_label"])
    ]
    sub["effect_text"] = [
        f"{fmt_num(b)} [{fmt_num(lo)}, {fmt_num(hi)}]"
        for b, lo, hi in zip(sub["coefficient"], sub["ci_low"], sub["ci_high"])
    ]
    sub["p_text"] = [fmt_p(p) for p in sub["p_value"]]
    return sub


def forest_xlim(sub: pd.DataFrame) -> tuple[float, float]:
    vals = sub[["coefficient", "ci_low", "ci_high"]].to_numpy(dtype=float).ravel()
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return (-1.0, 1.0)
    lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
    lo = min(lo, 0.0)
    hi = max(hi, 0.0)
    span = hi - lo
    pad = max(span * 0.13, 0.15)
    return lo - pad, hi + pad


def build_y_layout() -> tuple[list[str], dict[str, float], dict[str, tuple[float, float]], list[float], float, float]:
    """
    构建 y 轴布局。
    - 灰白背景的边界使用 predictor 分组边界；
    - 表格横线也使用同一组边界，解决二者位置不匹配的问题；
    - 默认让绘图区底部直接贴着最底部数据分组边界，减少下方留白。
    """
    plot_vars = VARIABLE_ORDER[::-1]
    y_base = {v: i * ROW_STEP for i, v in enumerate(plot_vars)}
    centers = [y_base[v] for v in plot_vars]

    separators = [(centers[i] + centers[i + 1]) / 2.0 for i in range(len(centers) - 1)]
    lower_outer = centers[0] - ROW_STEP / 2.0
    upper_outer = centers[-1] + ROW_STEP / 2.0

    if len(centers) == 1:
        group_bounds = {plot_vars[0]: (lower_outer, upper_outer)}
    else:
        bounds: dict[str, tuple[float, float]] = {}
        for i, var in enumerate(plot_vars):
            lo = lower_outer if i == 0 else separators[i - 1]
            hi = upper_outer if i == len(plot_vars) - 1 else separators[i]
            bounds[var] = (lo, hi)
        group_bounds = bounds

    if BOTTOM_ALIGN_TO_DATA_EDGE:
        # 让图件底部直接贴着最底部 Predictor 分组边界，而不是根据文字位置额外留白。
        # 顶部仍保留表头空间，避免 Urbanization Rate 组压到列名。
        y_min = lower_outer - BOTTOM_DATA_EDGE_PAD * ROW_STEP
        top_frac = min(TOP_DATA_EDGE_AXES_FRAC, HEADER_BAND_BOTTOM - 0.006)
        top_frac = max(top_frac, 0.55)
        y_max = y_min + (upper_outer - y_min) / top_frac
    else:
        max_offset = max(abs(v) for v in ACTIVITY_OFFSETS.values())
        lowest_text_y = centers[0] - max_offset
        highest_text_y = centers[-1] + max_offset

        # 反算 y 轴范围：让最低和最高活动行落在设定的轴域比例上。
        target_span = max(TOP_ACTIVITY_AXES_FRAC - BOTTOM_ACTIVITY_AXES_FRAC, 0.50)
        y_range = (highest_text_y - lowest_text_y) / target_span
        y_min = lowest_text_y - BOTTOM_ACTIVITY_AXES_FRAC * y_range
        y_max = y_min + y_range

    return plot_vars, y_base, group_bounds, separators, y_min, y_max


def add_header_band(
    ax: plt.Axes,
    labels: list[tuple[float, str, str, str]],
    *,
    fontname: str | None = None,
    draw_rules: bool = False,
) -> None:
    """添加表头灰底；粗灰线仅用于右侧表格，左侧森林图保持原代码外观。"""
    ax.axhspan(HEADER_BAND_BOTTOM, HEADER_BAND_TOP, transform=ax.transAxes, color=HEADER_BAND_COLOR, zorder=-3)
    text_kwargs = {} if fontname is None else {"fontname": fontname}
    for x, label, ha, color in labels:
        ax.text(
            x,
            HEADER_Y,
            label,
            transform=ax.transAxes,
            ha=ha,
            va="center",
            fontsize=HEADER_FONT_SIZE,
            fontweight=HEADER_FONT_WEIGHT,
            color=color,
            **text_kwargs,
        )
    if draw_rules:
        add_major_rules(ax)


def add_major_rules(ax: plt.Axes) -> None:
    """在列名下方和底部添加较粗的灰色实线。"""
    ax.plot(
        [0, 1],
        [MAJOR_RULE_HEADER_Y, MAJOR_RULE_HEADER_Y],
        transform=ax.transAxes,
        color=MAJOR_RULE_COLOR,
        lw=MAJOR_RULE_LW,
        solid_capstyle="butt",
        clip_on=False,
        zorder=20,
    )
    ax.plot(
        [0, 1],
        [MAJOR_RULE_BOTTOM_Y, MAJOR_RULE_BOTTOM_Y],
        transform=ax.transAxes,
        color=MAJOR_RULE_COLOR,
        lw=MAJOR_RULE_LW,
        solid_capstyle="butt",
        clip_on=False,
        zorder=20,
    )


def add_alternating_bands(
    fig: plt.Figure,
    axes: list[plt.Axes],
    ref_ax: plt.Axes,
    plot_vars: list[str],
    group_bounds: dict[str, tuple[float, float]],
) -> None:
    """
    绘制灰白背景。

    默认使用 figure 级矩形跨越左侧标签、森林图和右侧表格三块区域。
    这样可以避免每个 Axes 单独 axhspan 在 PNG/SVG/PDF 导出或图片查看器缩放时
    产生类似 Street Intersection Density 行中部的异常灰色横线。
    """
    if not SHOW_ALT_ROW_BANDS:
        return

    mode = str(ROW_BAND_DRAW_MODE).lower().strip()

    if mode == "figure":
        # 让 Axes 背景透明，使 figure 级背景矩形可以显示出来。
        for ax in axes:
            ax.set_facecolor(AXES_FACE_COLOR)
            ax.patch.set_alpha(0)

        left = min(ax.get_position().x0 for ax in axes) - ROW_BAND_X_PAD_FIG
        right = max(ax.get_position().x1 for ax in axes) + ROW_BAND_X_PAD_FIG
        width = right - left

        for i, variable in enumerate(plot_vars):
            draw_grey = (i % 2 == 0) if ALT_ROW_START_WITH_GREY else (i % 2 == 1)
            if not draw_grey:
                continue

            lo, hi = group_bounds[variable]
            lo -= ROW_BAND_Y_OVERLAP
            hi += ROW_BAND_Y_OVERLAP

            # 将数据坐标 y 转为 figure 坐标，绘制一整条连续背景带。
            y0_fig = fig.transFigure.inverted().transform(ref_ax.transData.transform((0, lo)))[1]
            y1_fig = fig.transFigure.inverted().transform(ref_ax.transData.transform((0, hi)))[1]
            y_bottom = min(y0_fig, y1_fig)
            height = abs(y1_fig - y0_fig)

            rect = Rectangle(
                (left, y_bottom),
                width,
                height,
                transform=fig.transFigure,
                facecolor=ALT_ROW_COLOR,
                edgecolor="none",
                linewidth=0,
                antialiased=False,
                zorder=ROW_BAND_ZORDER,
                clip_on=False,
            )
            fig.add_artist(rect)
        return

    # 回退模式：在每个 Axes 内分别绘制背景。通常不建议使用，可能在缩放时出现细线接缝。
    for i, variable in enumerate(plot_vars):
        draw_grey = (i % 2 == 0) if ALT_ROW_START_WITH_GREY else (i % 2 == 1)
        if not draw_grey:
            continue
        lo, hi = group_bounds[variable]
        lo -= ROW_BAND_Y_OVERLAP
        hi += ROW_BAND_Y_OVERLAP
        for ax in axes:
            ax.axhspan(
                lo,
                hi,
                facecolor=ALT_ROW_COLOR,
                edgecolor="none",
                linewidth=0,
                antialiased=False,
                zorder=-5,
            )

def add_group_separators(ax: plt.Axes, separators: list[float]) -> None:
    """仅在右侧表格区域添加 Predictor 大类之间的灰色实线。"""
    if not SHOW_TABLE_GROUP_SEPARATOR:
        return
    for sep_y in separators:
        ax.axhline(
            sep_y,
            color=GROUP_SEPARATOR_COLOR,
            lw=GROUP_SEPARATOR_LW,
            ls="-",
            zorder=-1,
            clip_on=False,
        )


def table_column_layout() -> dict[str, dict[str, float]]:
    """
    计算右侧表格三列的列位置。

    返回每列的 left / center / right / text_x：
    - left/right/center 用于表头或后续需要画列边界时使用；
    - text_x 是数据文字的推荐位置。
    """
    ratios = np.asarray(TABLE_COLUMN_WIDTH_RATIOS, dtype=float)
    if ratios.size != 3 or np.any(~np.isfinite(ratios)) or np.any(ratios <= 0):
        raise ValueError("TABLE_COLUMN_WIDTH_RATIOS 必须是 3 个大于 0 的数字，例如 (1.0, 1.0, 1.0)。")

    widths = ratios / ratios.sum()
    lefts = np.r_[0.0, np.cumsum(widths)[:-1]]
    rights = np.cumsum(widths)
    centers = (lefts + rights) / 2.0

    return {
        "activity": {
            "left": float(lefts[0]),
            "center": float(centers[0]),
            "right": float(rights[0]),
            "text_x": float(lefts[0] + TABLE_COLUMN_INNER_PAD),
        },
        "beta": {
            "left": float(lefts[1]),
            "center": float(centers[1]),
            "right": float(rights[1]),
            "text_x": float(lefts[1] + TABLE_COLUMN_INNER_PAD),
        },
        "p": {
            "left": float(lefts[2]),
            "center": float(centers[2]),
            "right": float(rights[2]),
            "text_x": float(centers[2]),
        },
    }


def table_header_specs() -> list[tuple[float, str, str, str]]:
    """生成右侧表格表头位置。"""
    cols = table_column_layout()
    if TABLE_HEADER_CENTERED_IN_EQUAL_COLUMNS:
        return [
            (cols["activity"]["center"], "Activity", "center", HEADER_TEXT_COLOR),
            (cols["beta"]["center"], "beta [95% CI]", "center", HEADER_TEXT_COLOR),
            (cols["p"]["center"], "p", "center", HEADER_TEXT_COLOR),
        ]

    return [
        (cols["activity"]["text_x"], "Activity", "left", HEADER_TEXT_COLOR),
        (cols["beta"]["text_x"], "beta [95% CI]", "left", HEADER_TEXT_COLOR),
        (cols["p"]["center"], "p", "center", HEADER_TEXT_COLOR),
    ]


def table_text_kwargs() -> dict[str, object]:
    return {
        "color": TABLE_TEXT_COLOR,
        "fontsize": TABLE_FONT_SIZE,
        "fontname": TABLE_FONT_FAMILY,
    }


def save_right_table_csv(sub: pd.DataFrame, mode: str, panel_letter: str) -> Path:
    """单独导出图件右侧表格所显示的结果。

    右侧表格显示三列：Activity、beta [95% CI] 和 p。
    为便于复核和后续排版，CSV 同时保留 coefficient / ci_low / ci_high / p_value 等数值列。
    """
    TABLE_CSV_DIR.mkdir(parents=True, exist_ok=True)

    out = drop_direction_columns(sub).copy()
    out["panel"] = panel_letter
    out["mode_label"] = MODES.get(mode, mode)
    out["predictor"] = out["variable_only"].astype(str)
    out["activity_label"] = out["activity_type"].astype(str).map(ACTIVITY_LABELS).fillna(out["activity_type"].astype(str))
    out["beta_95ci"] = out["effect_text"]
    out["p_display"] = out["p_text"]

    preferred_cols = [
        "panel",
        "mode",
        "mode_label",
        "predictor",
        "activity_type",
        "activity_label",
        "coefficient",
        "ci_low",
        "ci_high",
        "beta_95ci",
        "p_value",
        "p_display",
        "significance_label",
        "partition",
        "indicator",
        "base_model",
    ]
    preferred_cols = [c for c in preferred_cols if c in out.columns]
    other_cols = [c for c in out.columns if c not in preferred_cols]
    out = out[preferred_cols + other_cols]

    table_csv = TABLE_CSV_DIR / f"panel_{panel_letter}_{mode.lower()}_right_table_results.csv"
    out.to_csv(table_csv, index=False, encoding="utf-8-sig")
    return table_csv


# =============================================================================
# 5. 绘图主函数
# =============================================================================
def plot_panel(mode: str, panel_letter: str) -> tuple[Path, Path, Path, Path]:
    sub = prepare_panel_data(mode)

    # 输出绘图筛选后的完整数据表；同步删除 direction 列。
    panel_csv = DATA_DIR / f"panel_{panel_letter}_{mode.lower()}_national_cehwi_composite_records.csv"
    panel_csv.parent.mkdir(parents=True, exist_ok=True)
    drop_direction_columns(sub).to_csv(panel_csv, index=False, encoding="utf-8-sig")

    # 将右侧表格显示内容单独导出到截图三路径。
    right_table_csv = save_right_table_csv(sub, mode, panel_letter)

    n_vars = len(VARIABLE_ORDER)
    plot_vars, y_base, group_bounds, separators, y_min, y_max = build_y_layout()
    xlim = forest_xlim(sub)

    fig = plt.figure(figsize=figure_size(n_vars))

    # 通过两个顶部参数控制宽度：
    # 1) PREDICTOR_TO_FOREST_WIDTH_RATIO 控制左侧图内部“行名列 : 森林图列”；
    # 2) LEFT_PLOT_TO_RIGHT_TABLE_WIDTH_RATIO 控制“左侧图整体 : 右侧表格”。
    label_part, forest_part = PREDICTOR_TO_FOREST_WIDTH_RATIO
    left_plot_part, table_part = LEFT_PLOT_TO_RIGHT_TABLE_WIDTH_RATIO

    if label_part <= 0 or forest_part <= 0:
        raise ValueError("PREDICTOR_TO_FOREST_WIDTH_RATIO 中两个数值都必须大于 0。")
    if left_plot_part <= 0 or table_part <= 0:
        raise ValueError("LEFT_PLOT_TO_RIGHT_TABLE_WIDTH_RATIO 中两个数值都必须大于 0。")

    left_label_ratio = left_plot_part * label_part
    left_forest_ratio = left_plot_part * forest_part
    table_ratio = table_part

    gs = fig.add_gridspec(
        1,
        3,
        width_ratios=[left_label_ratio, left_forest_ratio, table_ratio],
        left=GRID_LEFT,
        right=GRID_RIGHT,
        top=GRID_TOP,
        bottom=GRID_BOTTOM,
        wspace=GRID_WSPACE,
    )
    ax_label = fig.add_subplot(gs[0, 0])
    ax_forest = fig.add_subplot(gs[0, 1], sharey=ax_label)
    ax_table = fig.add_subplot(gs[0, 2], sharey=ax_label)
    axes = [ax_label, ax_forest, ax_table]

    for ax in axes:
        ax.set_ylim(y_min, y_max)
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    add_alternating_bands(fig, axes, ax_label, plot_vars, group_bounds)
    add_group_separators(ax_table, separators)

    for variable in plot_vars:
        y = y_base[variable]
        weight = HIGHLIGHT_LABEL_FONT_WEIGHT if variable in HIGHLIGHT_VARIABLES else NORMAL_LABEL_FONT_WEIGHT
        ax_label.text(
            0.98,
            y,
            variable,
            ha="right",
            va="center",
            fontsize=LABEL_FONT_SIZE,
            fontweight=weight,
            color=PREDICTOR_TEXT_COLOR,
        )

    ax_label.set_xlim(0, 1)
    ax_label.set_xticks([])
    add_header_band(ax_label, [(0.98, "Predictor", "right", HEADER_TEXT_COLOR)], draw_rules=False)

    ax_forest.spines["bottom"].set_visible(True)
    ax_forest.axvline(0, color=ZERO_LINE_COLOR, lw=ZERO_LINE_WIDTH, ls=ZERO_LINE_STYLE, zorder=0)
    ax_forest.grid(axis="x", color=GRID_COLOR, lw=GRID_LINE_WIDTH)
    ax_forest.set_xlim(*xlim)
    ax_forest.set_xlabel("Meta-regression coefficient", labelpad=4, fontsize=AXIS_FONT_SIZE)
    ax_forest.tick_params(axis="x", labelsize=TICK_FONT_SIZE)
    add_header_band(ax_forest, [(0.02, "Coefficient (95% CI)", "left", HEADER_TEXT_COLOR)], draw_rules=False)

    ax_table.set_xlim(0, 1)
    ax_table.set_xticks([])
    add_header_band(
        ax_table,
        table_header_specs(),
        fontname=TABLE_FONT_FAMILY,
        draw_rules=True,
    )

    table_kwargs = table_text_kwargs()
    table_cols = table_column_layout()
    for _, r in sub.iterrows():
        activity = str(r["activity_type"])
        variable = str(r["variable_only"])
        y = y_base[variable] + ACTIVITY_OFFSETS[activity]
        color = ACTIVITY_COLORS[activity]

        ax_forest.hlines(
            y,
            r["ci_low"],
            r["ci_high"],
            color=color,
            lw=CI_LINE_WIDTH,
            alpha=CI_LINE_ALPHA,
            zorder=2,
            clip_on=False,
        )
        ax_forest.scatter(
            r["coefficient"],
            y,
            s=POINT_SIZE,
            color=color,
            edgecolor=POINT_EDGE_COLOR,
            linewidth=POINT_EDGE_WIDTH,
            zorder=4,
            clip_on=False,
        )
        if SHOW_SIGNIFICANCE_STARS_ON_FOREST and str(r["significance_label"]) != "ns":
            ax_forest.text(
                r["coefficient"] + (xlim[1] - xlim[0]) * STAR_X_OFFSET_FRAC,
                y + STAR_Y_OFFSET,
                str(r["significance_label"]),
                ha="left",
                va="bottom",
                fontsize=STAR_FONT_SIZE,
                color=color,
                fontweight="bold",
                clip_on=False,
            )

        # 右侧表格全部使用黑色 Times New Roman；direction 和 sig. 列均不再显示。
        ax_table.text(
            table_cols["activity"]["text_x"],
            y,
            ACTIVITY_LABELS[activity],
            ha=TABLE_ACTIVITY_HA,
            va="center",
            **table_kwargs,
        )
        ax_table.text(
            table_cols["beta"]["text_x"],
            y,
            str(r["effect_text"]),
            ha=TABLE_BETA_HA,
            va="center",
            **table_kwargs,
        )
        ax_table.text(
            table_cols["p"]["text_x"],
            y,
            str(r["p_text"]),
            ha=TABLE_P_HA,
            va="center",
            **table_kwargs,
        )

    if SHOW_LEGEND:
        handles = [
            Line2D(
                [0],
                [0],
                marker="o",
                lw=LEGEND_LINE_WIDTH,
                color=ACTIVITY_COLORS[a],
                markersize=LEGEND_MARKER_SIZE,
                label=ACTIVITY_LABELS[a],
            )
            for a in ACTIVITY_ORDER
        ]
        fig.legend(
            handles=handles,
            loc="upper right",
            bbox_to_anchor=(LEGEND_X, LEGEND_Y),
            ncol=LEGEND_NCOL,
            fontsize=LEGEND_FONT_SIZE,
            handlelength=LEGEND_HANDLE_LENGTH,
            columnspacing=LEGEND_COLUMN_SPACING,
        )

    title = f"{panel_letter}) National | CEHWI | Composite | {MODES[mode]}"
    subtitle = "Built-environment meta-regression forest with aligned coefficient table; Building Density is BD, not BH."
    fig.text(
        TITLE_X,
        TITLE_Y,
        title,
        ha="left",
        va="top",
        fontsize=TITLE_FONT_SIZE,
        fontweight=TITLE_FONT_WEIGHT,
    )
    if SHOW_SUBTITLE:
        fig.text(
            SUBTITLE_X,
            SUBTITLE_Y,
            subtitle,
            ha="left",
            va="top",
            fontsize=SUBTITLE_FONT_SIZE,
            color=SUBTITLE_COLOR,
        )

    stem = FIG_DIR / f"panel_{panel_letter}_{mode.lower()}_national_cehwi_composite_forest_table"
    stem.parent.mkdir(parents=True, exist_ok=True)
    png = stem.with_suffix(".png")
    svg = stem.with_suffix(".svg")

    save_kwargs = {"bbox_inches": "tight", "pad_inches": EXPORT_PAD_INCHES} if SAVE_BBOX_TIGHT else {}
    fig.savefig(png, dpi=EXPORT_DPI, **save_kwargs)
    fig.savefig(svg, **save_kwargs)
    plt.close(fig)
    return png, svg, panel_csv, right_table_csv


# =============================================================================
# 6. 数据归档与说明文件
# =============================================================================
def audit_bd_bh() -> tuple[Path, Path]:
    """
    优先使用图1 data 文件夹中已有的 bd_bh_source_audit_paths.csv。
    原代码需要从旧 RESULT_ROOT 递归扫描 city_level_covariates_SUMMARY.csv；
    现在不再依赖旧 D 盘 R 输出目录。
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    audit_path = DATA_DIR / "bd_bh_source_audit_paths.csv"
    summary_path = DATA_DIR / "bd_bh_source_audit_summary.csv"

    if SOURCE_AUDIT_PATHS.exists():
        audit = pd.read_csv(SOURCE_AUDIT_PATHS)
        drop_direction_columns(audit).to_csv(audit_path, index=False, encoding="utf-8-sig")
    else:
        audit = pd.DataFrame(columns=["source_summary", "variable_only", "n_rows"])
        audit.to_csv(audit_path, index=False, encoding="utf-8-sig")
        print(f"Warning: 未找到 {SOURCE_AUDIT_PATHS}，已生成空的 BD/BH audit 文件。")

    if not audit.empty and {"variable_only", "source_summary", "n_rows"}.issubset(audit.columns):
        summary = (
            audit.groupby("variable_only", dropna=False)
            .agg(n_source_files=("source_summary", "nunique"), n_rows=("n_rows", "sum"))
            .reset_index()
            .sort_values(["variable_only"])
        )
    else:
        summary = pd.DataFrame(columns=["variable_only", "n_source_files", "n_rows"])

    drop_direction_columns(summary).to_csv(summary_path, index=False, encoding="utf-8-sig")
    return audit_path, summary_path


def copy_if_exists(src: Path, dst_dir: Path) -> None:
    if src.exists():
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / src.name
        try:
            if src.resolve() == dst.resolve():
                return
        except OSError:
            pass

        # 复制 CSV 时同步删除 direction 列；其他文件按原样复制。
        if src.suffix.lower() == ".csv":
            try:
                df = pd.read_csv(src)
                drop_direction_columns(df).to_csv(dst, index=False, encoding="utf-8-sig")
                return
            except Exception as exc:
                print(f"Warning: 读取 CSV 失败，将按原文件复制：{src}\n  {exc}")
        shutil.copy2(src, dst)


def backup_source_data() -> None:
    """把图1和图2的关键 data 文件复制到输出 data 目录中，便于结果复现；CSV 备份同步删除 direction 列。"""
    for src in [SOURCE_TABLE, SOURCE_AUDIT_PATHS, SOURCE_PANEL_C, SOURCE_PANEL_D, *LAG_PROFILE_FILES]:
        copy_if_exists(src, DATA_DIR)


def write_readme(outputs: list[tuple[str, Path, Path, Path, Path]], audit_path: Path, summary_path: Path) -> None:
    lines = [
        "# Panel a/b BE forest table redraw",
        "",
        "Scope: NATIONAL, CEHWI, composite heatwave model; source is the final no-crime/no-BH grouped BE meta-regression table.",
        "",
        "Target output directory:",
        f"- `{OUT_ROOT}`",
        "",
        "Input data paths:",
        f"- Source data directory: `{SOURCE_DATA_DIR}`",
        f"- Forest-table source: `{SOURCE_TABLE}`",
        f"- BD/BH audit source: `{SOURCE_AUDIT_PATHS}`",
        f"- Figure output directory: `{FIG_DIR}`",
        f"- Right-table CSV output directory: `{TABLE_CSV_DIR}`",
        "",
        "Style changes:",
        "- All visualization parameters are collected near the top of the script.",
        "- Two width-ratio parameters were added: predictor-label versus forest-plot width, and left-plot versus right-table width.",
        "- The right-side table now uses equal-width columns by default through TABLE_COLUMN_WIDTH_RATIOS = (1.0, 1.0, 1.0).",
        "- Figure height is adaptive to the number of predictor groups in the right-side table.",
        "- Grey-white row bands are drawn as one continuous figure-level background to avoid abnormal seam lines; group separator lines use the same predictor-group boundaries.",
        "- Bottom whitespace is minimized by aligning the lower y-axis limit with the bottom predictor-group edge.",
        "- Title and table fonts are enlarged.",
        "- The table text is black and uses Times New Roman when the font is available.",
        "- The direction and sig. columns have been removed from the right-side table; it now shows activity, beta [95% CI], and p value only.",
        "",
        "Lag-profile note:",
        "- The fig5d lag-response CSV files are copied into the output data directory for record keeping.",
        "- This panel a/b forest-table script does not directly use the lag-response CSV files when drawing the forest plot.",
        "",
        "BD/BH audit:",
        "- The script now reads the existing BD/BH audit CSV from the Fig5ab data directory instead of scanning the old D-drive R-output directory.",
        "- If the audit source file is missing, the script writes an empty audit table and continues.",
        "",
        "Outputs:",
    ]
    for label, png, svg, csv_path, table_csv_path in outputs:
        lines.append(f"- {label}: `{png}` / `{svg}` / plotting data `{csv_path}` / right-table CSV `{table_csv_path}`")
    lines.extend(
        [
            f"- BD/BH audit paths: `{audit_path}`",
            f"- BD/BH audit summary: `{summary_path}`",
            "",
            "Columns shown in the right-side table: activity, beta [95% CI], p value.",
        ]
    )
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    (REPORT_DIR / "README_panel_cd_redraw.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


# =============================================================================
# 7. 主程序
# =============================================================================
def main() -> None:
    validate_inputs()
    apply_style()
    for d in [FIG_DIR, TABLE_CSV_DIR, DATA_DIR, CODE_DIR, REPORT_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    audit_path, summary_path = audit_bd_bh()
    outputs = []
    for panel_letter, mode in [("a", "MEAN_NO_CRIME_ACTIVITY_3PLUS1"), ("b", "GINI_NO_CRIME_ACTIVITY_3PLUS1")]:
        png, svg, csv_path, table_csv_path = plot_panel(mode, panel_letter)
        outputs.append((MODES[mode], png, svg, csv_path, table_csv_path))

    backup_source_data()

    # 在 .py 文件中运行时，__file__ 存在，可以把当前脚本复制到输出 code 文件夹。
    # 在 Jupyter / Spyder cell / Notebook 中运行时，__file__ 通常不存在，因此这里跳过该复制步骤，避免 NameError。
    if "__file__" in globals():
        copy_if_exists(Path(__file__), CODE_DIR)
    else:
        (CODE_DIR / "NOTE_running_in_notebook.txt").write_text(
            "本次代码是在 Jupyter/Notebook/交互式环境中运行的，因此不存在 __file__，未复制当前 .py 脚本。\n"
            "The canonical notebook is already archived in this module's code directory.\n",
            encoding="utf-8",
        )

    write_readme(outputs, audit_path, summary_path)

    print(f"Output root: {OUT_ROOT}")
    for label, png, svg, csv_path, table_csv_path in outputs:
        print(f"{label}: {png} | {svg} | plotting data: {csv_path} | right-table CSV: {table_csv_path}")
    print(f"Audit: {summary_path}")


if __name__ == "__main__":
    main()